# M2 — Feature Pipeline Visualisation

Shows what the leakage-safe feature pipeline actually produces and proves visually that no future information sneaks in. Every panel here doubles as something a recruiter can glance at and understand in 30 seconds.

**What you should see:**
- 48 features × 96 quarter-hours per delivery day, 99.98% of cells finite.
- The TSO forecast for day D is fully known at issue time T (legitimate, not leakage).
- Population-weighted holiday fraction picks up Bundesland-specific drops the federal-only flag misses.
- The corrupt-future probe makes leakage-safety visible: scrambling post-issue data leaves features unchanged.

In [ ]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from loadforecast.backtest import issue_time_for, load_smard_15min
from loadforecast.features import build_target_day_features, target_residual
from loadforecast.features.availability import RULES, classify_column
from loadforecast.features.calendar import (
    is_bridge_day,
    is_federal_holiday,
    population_weighted_holiday_fraction,
)

PARQUET = Path("../smard_merged_15min.parquet")
assert PARQUET.exists(), "Run from the notebooks/ folder."
df = load_smard_15min(PARQUET)
print(f"Loaded {df.shape[0]:,} rows, {df.shape[1]} cols.")

## 1. The availability rules — what's known when?

For a forecaster issuing predictions at **T = D-1 12:00 Europe/Berlin**, three column types have three different cut-offs:

In [ ]:
rule_table = pd.DataFrame(
    [
        {
            "kind": kind,
            "max_age_offset": str(rule.max_age_offset),
            "interpretation": {
                "actual": "Measured. Usable only for ts < T.",
                "forecast": "TSO publishes day D forecast at D-1 ~10:00. Usable for ts < T + 48h.",
                "price": "Day-ahead prices clear at D-1 ~12:45 (after T). Usable for ts < T - 12h.",
                "other": "Default to strictest rule.",
            }[kind],
        }
        for kind, rule in RULES.items()
    ]
)
rule_table

In [ ]:
# Visualise the cutoffs around an example issue time.
# We pass timestamps as ISO strings to plotly because in pandas 2.2 plotly's
# add_vline internally tries Timestamp +/- int and trips a TypeError.
DELIVERY = date(2025, 9, 16)
issue = issue_time_for(DELIVERY)
issue_local = issue.tz_convert("Europe/Berlin")
issue_iso = issue.tz_convert("UTC").tz_localize(None).isoformat()

rows = []
for kind, rule in RULES.items():
    cutoff = (issue + rule.max_age_offset).tz_convert("UTC").tz_localize(None)
    start = (issue.tz_convert("UTC").tz_localize(None) - pd.Timedelta(days=2)).isoformat()
    rows.append({
        "kind": kind,
        "start": start,
        "cutoff": cutoff.isoformat(),
        "label_at": (issue + rule.max_age_offset).tz_convert("Europe/Berlin").strftime("%a %H:%M"),
    })
rule_df = pd.DataFrame(rows)

color_map = {"actual": "#1f77b4", "forecast": "#2ca02c", "price": "#d62728", "other": "#7f7f7f"}
order = ["actual", "forecast", "price", "other"]

fig = go.Figure()
for _, r in rule_df.iterrows():
    fig.add_trace(go.Scatter(
        x=[r["start"], r["cutoff"]],
        y=[r["kind"], r["kind"]],
        mode="lines",
        line=dict(color=color_map[r["kind"]], width=20),
        name=f"{r['kind']} - usable until {r['label_at']}",
        hovertemplate=f"{r['kind']}: usable until {r['label_at']}<extra></extra>",
    ))
# Issue-time marker as a vertical Scatter, not add_vline
fig.add_trace(go.Scatter(
    x=[issue_iso, issue_iso],
    y=[order[0], order[-1]],
    mode="lines",
    line=dict(color="black", width=2, dash="dot"),
    name="Issue T = D-1 12:00 Berlin",
    hovertemplate="Issue T<extra></extra>",
))
fig.update_layout(
    title=f"Availability cut-offs around issue T = {issue_local.strftime('%a %d %b %Y %H:%M Berlin')}",
    xaxis_title="Time (UTC, naive)",
    yaxis=dict(categoryorder="array", categoryarray=order, title="Column kind"),
    height=320,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.35),
)
fig

## 2. One delivery day's feature matrix

In [ ]:
feats = build_target_day_features(df, issue)
y_resid = target_residual(df, issue)
print("shape:", feats.shape)
print("finite cells:", feats.size - feats.isna().sum().sum(), "/", feats.size)
feats.head()

In [ ]:
# Heatmap — z-scored so all features fit on one colour scale
z = (feats - feats.mean()) / feats.std().replace(0, np.nan)
fig = go.Figure(
    go.Heatmap(
        z=z.values.T,
        x=feats.index,
        y=feats.columns,
        colorscale="RdBu",
        zmid=0,
        zmin=-3,
        zmax=3,
        colorbar=dict(title="z"),
    )
)
fig.update_layout(
    title=f"Feature matrix — delivery day {DELIVERY} (z-scored)",
    xaxis_title="Target quarter-hour (UTC)",
    yaxis_title="Feature",
    height=900,
)
fig

## 3. The TSO forecast and the residual target — the same day

In [ ]:
actual = df.loc[feats.index, "actual_cons__grid_load"]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.05)
fig.add_trace(go.Scatter(x=feats.index, y=actual, name="Actual load", line=dict(color="black", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=feats.index, y=feats["tso_load_fc"], name="TSO forecast", line=dict(color="#1f77b4", dash="dash")), row=1, col=1)
fig.add_trace(go.Bar(x=feats.index, y=y_resid, name="Residual = actual - TSO", marker_color="#d62728"), row=2, col=1)
fig.update_yaxes(title_text="Load (MW)", row=1, col=1)
fig.update_yaxes(title_text="Residual (MW)", row=2, col=1)
fig.update_layout(
    title=f"Delivery day {DELIVERY} — what the model is asked to learn (the bottom panel)",
    height=550,
    hovermode="x unified",
)
fig

## 4. Population-weighted holiday fraction across 2024

A boolean `is_federal_holiday` misses Corpus Christi (~28% of pop on holiday), Reformation Day (~30%), All Saints' Day (~40%), etc. The weighted fraction captures the consumption impact properly.

In [ ]:
year_idx = pd.date_range("2024-01-01", "2024-12-31", freq="D", tz="Europe/Berlin").tz_convert("UTC")
frac = population_weighted_holiday_fraction(year_idx)
fed = is_federal_holiday(year_idx).astype(int)
bridge = is_bridge_day(year_idx).astype(int)

fig = go.Figure()
fig.add_trace(go.Bar(x=year_idx, y=frac, name="Pop-weighted holiday fraction", marker_color="#1f77b4"))
fig.add_trace(go.Bar(x=year_idx, y=fed * 0.05 - 0.06, name="Federal holiday flag", marker_color="#2ca02c"))
fig.add_trace(go.Bar(x=year_idx, y=bridge * 0.05 - 0.12, name="Bridge day", marker_color="#ff7f0e"))
fig.update_layout(
    title="German calendar features — 2024",
    xaxis_title="Date",
    yaxis_title="Holiday fraction (top) / flags (bottom)",
    barmode="overlay",
    height=400,
)
fig

## 5. Where do features correlate with the residual target?

Compute features + residual target across many delivery days, then correlate. Features at the top of this chart are the ones an ML model will lean on hardest.

In [ ]:
# Sample 60 random delivery days from 2024 to keep this fast
rng = np.random.default_rng(42)
all_days = pd.date_range("2024-01-15", "2024-12-15", freq="D").date
sample_days = rng.choice(all_days, size=60, replace=False)
sample_days = sorted(sample_days)

X_all, y_all = [], []
for d in sample_days:
    issue_d = issue_time_for(d)
    fx = build_target_day_features(df, issue_d)
    yr = target_residual(df, issue_d)
    if yr.isna().any():
        continue
    X_all.append(fx)
    y_all.append(yr)
X = pd.concat(X_all)
y = pd.concat(y_all)
print(f"Stacked: {X.shape[0]} rows ({X.shape[0] // 96} days) x {X.shape[1]} features")

corrs = X.corrwith(y).dropna().sort_values(key=lambda s: s.abs(), ascending=True)
fig = go.Figure(go.Bar(
    x=corrs.values, y=corrs.index, orientation="h",
    marker_color=["#d62728" if v < 0 else "#1f77b4" for v in corrs.values],
))
fig.update_layout(
    title="Pearson correlation of each feature with the residual target (60 days, 2024)",
    xaxis_title="Correlation with (actual_load - TSO_forecast)",
    height=900,
)
fig

## 6. Leakage-safety probe — visual proof

In [ ]:
# Build features on the clean df, then on a deliberately-corrupted df, and overlay.
PROBE_DELIVERY = date(2025, 9, 16)
issue_p = issue_time_for(PROBE_DELIVERY)
feats_clean = build_target_day_features(df, issue_p)

corrupt = df.copy()
rng = np.random.default_rng(0)
for col in corrupt.columns:
    kind = classify_column(col)
    cutoff = issue_p + RULES[kind].max_age_offset
    mask = corrupt.index >= cutoff
    if mask.any():
        corrupt.loc[mask, col] = rng.normal(loc=1e9, scale=1e9, size=mask.sum())

feats_corrupt = build_target_day_features(corrupt, issue_p)
diff = (feats_clean - feats_corrupt).abs()
max_diff = diff.max().max()
print(f"Largest absolute difference across all 96 x 48 = 4608 cells: {max_diff:.6g}")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Clean features (z-scored)", "Features after post-issue corruption"))
fig.add_trace(go.Heatmap(z=((feats_clean - feats_clean.mean()) / feats_clean.std().replace(0, np.nan)).values.T, colorscale="RdBu", zmid=0, zmin=-3, zmax=3, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=((feats_corrupt - feats_corrupt.mean()) / feats_corrupt.std().replace(0, np.nan)).values.T, colorscale="RdBu", zmid=0, zmin=-3, zmax=3, showscale=False), row=1, col=2)
fig.update_layout(
    title=f"Corrupt-future probe — features identical (max diff {max_diff:.2g})",
    height=700,
)
fig

## 7. The bar to beat

If a model trained on these 48 features can predict the residual target with MAE meaningfully below the residual's own standard deviation, it will beat the TSO baseline.

In [ ]:
print(f"Residual target — sample of {len(y):,} cells across {len(y) // 96} delivery days")
print(f"  mean        : {y.mean():+.1f} MW")
print(f"  std         : {y.std():.1f} MW")
print(f"  MAE if predicting zero: {y.abs().mean():.1f} MW   <- the floor any model must beat")
print()
print("From M1 we know the TSO MAE on the long evaluation window is ~505 MW.")
print("A model that predicts residuals with MAE < 505 MW reduces total error vs TSO.")